# 🌾 Intelligent Crop Advisory System (ICAS)
### AI-Powered Crop Recommendation for Kenyan Smallholder Farmers

**Author:** Ryan Nduati — Strathmore University, Faculty of Computing

---

This notebook contains the complete data science pipeline:
1. **Data Generation & Collection** — Synthetic Kenyan agricultural dataset
2. **Exploratory Data Analysis (EDA)** — Statistical summaries & visualizations
3. **Feature Engineering** — Preprocessing for ML models
4. **Model Training** — Random Forest & XGBoost classifiers
5. **Model Evaluation** — Accuracy, F1-score, confusion matrix, cross-validation
6. **Prediction & Export** — Make predictions and export model for deployment

## 📦 Step 0: Install & Import Dependencies

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn joblib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from xgboost import XGBClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 6)
print('✅ All libraries imported successfully!')

---
## 📊 Step 1: Data Generation — Kenyan Agricultural Dataset

We generate a realistic synthetic dataset modeled after Kenyan agricultural conditions including:
- **Soil types**: Clay, Sandy, Loam, Silt, Red, Black Cotton
- **Regions**: Major Kenyan agricultural zones
- **Climate features**: Temperature, rainfall, humidity
- **Soil chemistry**: pH, nitrogen, phosphorus, potassium
- **Target**: Best crop to grow

In [ ]:
np.random.seed(42)
n_samples = 2000

# Kenyan regions with typical conditions
regions = {
    'Central Kenya (Nyeri, Murang\'a)': {'temp': (14, 22), 'rain': (900, 1600), 'humidity': (60, 80)},
    'Rift Valley (Nakuru, Uasin Gishu)': {'temp': (12, 24), 'rain': (800, 1400), 'humidity': (55, 75)},
    'Western Kenya (Kakamega, Bungoma)': {'temp': (18, 28), 'rain': (1200, 2000), 'humidity': (65, 85)},
    'Coast (Mombasa, Kilifi)': {'temp': (24, 32), 'rain': (600, 1200), 'humidity': (70, 90)},
    'Eastern (Machakos, Kitui)': {'temp': (18, 28), 'rain': (400, 800), 'humidity': (40, 65)},
    'Nairobi Metropolitan': {'temp': (16, 26), 'rain': (700, 1100), 'humidity': (50, 70)},
    'North Rift (Trans Nzoia)': {'temp': (14, 24), 'rain': (1000, 1800), 'humidity': (60, 80)},
    'Nyanza (Kisumu, Homa Bay)': {'temp': (20, 30), 'rain': (800, 1500), 'humidity': (60, 80)},
}

soil_types = ['Clay', 'Sandy', 'Loam', 'Silt', 'Red', 'Black Cotton']

# Crop suitability rules
crops = ['Maize', 'Tea', 'Coffee', 'Wheat', 'Rice', 'Beans', 'Sorghum',
         'Sweet Potato', 'Cassava', 'Sugarcane', 'Millet', 'Groundnuts']

data = []
for _ in range(n_samples):
    region_name = np.random.choice(list(regions.keys()))
    region = regions[region_name]

    temp = np.random.uniform(*region['temp'])
    rain = np.random.uniform(*region['rain'])
    humidity = np.random.uniform(*region['humidity'])
    soil = np.random.choice(soil_types)

    # Soil chemistry
    ph = np.random.uniform(4.5, 8.5)
    nitrogen = np.random.uniform(10, 140)
    phosphorus = np.random.uniform(5, 80)
    potassium = np.random.uniform(5, 200)
    farm_size_acres = np.random.uniform(0.5, 20)
    altitude_m = np.random.uniform(0, 2800)

    # Rule-based crop assignment (simplified agronomic logic)
    if rain > 1400 and temp < 20 and ph > 4.5 and ph < 6.0:
        crop = 'Tea'
    elif rain > 800 and rain < 1500 and temp > 15 and temp < 25 and ph > 5.0 and ph < 7.0:
        crop = 'Coffee'
    elif rain > 1200 and temp > 22 and soil in ['Clay', 'Black Cotton', 'Silt']:
        crop = 'Rice'
    elif rain > 1000 and temp > 20 and soil in ['Loam', 'Clay', 'Black Cotton']:
        crop = 'Sugarcane'
    elif temp < 22 and rain > 600 and rain < 1200 and soil in ['Loam', 'Red', 'Black Cotton']:
        crop = 'Wheat'
    elif rain < 600 and temp > 22:
        crop = 'Sorghum'
    elif rain < 500 and soil in ['Sandy', 'Red']:
        crop = 'Millet'
    elif soil == 'Sandy' and rain > 600:
        crop = 'Groundnuts'
    elif rain > 600 and soil in ['Loam', 'Red']:
        crop = 'Beans'
    elif rain < 800 and soil in ['Sandy', 'Loam']:
        crop = 'Cassava'
    elif humidity > 60 and temp > 18:
        crop = 'Sweet Potato'
    else:
        crop = 'Maize'

    data.append({
        'region': region_name,
        'soil_type': soil,
        'temperature_c': round(temp, 1),
        'rainfall_mm': round(rain, 1),
        'humidity_pct': round(humidity, 1),
        'soil_ph': round(ph, 2),
        'nitrogen_kg_ha': round(nitrogen, 1),
        'phosphorus_kg_ha': round(phosphorus, 1),
        'potassium_kg_ha': round(potassium, 1),
        'farm_size_acres': round(farm_size_acres, 2),
        'altitude_m': round(altitude_m, 0),
        'recommended_crop': crop
    })

df = pd.DataFrame(data)
print(f'✅ Dataset created: {df.shape[0]} samples, {df.shape[1]} features')
df.head(10)

In [ ]:
# Save dataset to CSV
df.to_csv('kenya_crop_dataset.csv', index=False)
print('✅ Dataset saved to kenya_crop_dataset.csv')

---
## 🔍 Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('='*60)
print('DATASET SUMMARY STATISTICS')
print('='*60)
print(f'\nShape: {df.shape}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nNumerical Stats:')
df.describe().round(2)

In [ ]:
# Crop distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

crop_counts = df['recommended_crop'].value_counts()
colors = sns.color_palette('viridis', len(crop_counts))

axes[0].barh(crop_counts.index, crop_counts.values, color=colors)
axes[0].set_title('Crop Distribution in Dataset', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Count')

axes[1].pie(crop_counts.values, labels=crop_counts.index, autopct='%1.1f%%',
            colors=colors, pctdistance=0.85)
centre = plt.Circle((0,0), 0.70, fc='white')
axes[1].add_artist(centre)
axes[1].set_title('Crop Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('crop_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap for numerical features
numerical_cols = ['temperature_c', 'rainfall_mm', 'humidity_pct', 'soil_ph',
                  'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha',
                  'farm_size_acres', 'altitude_m']

fig, ax = plt.subplots(figsize=(12, 8))
correlation = df[numerical_cols].corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Rainfall vs Temperature scatter by crop
fig, ax = plt.subplots(figsize=(14, 8))
for crop in df['recommended_crop'].unique():
    subset = df[df['recommended_crop'] == crop]
    ax.scatter(subset['rainfall_mm'], subset['temperature_c'],
               label=crop, alpha=0.6, s=30)
ax.set_xlabel('Rainfall (mm/year)', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title('Crop Suitability: Rainfall vs Temperature', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('rainfall_vs_temp.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Soil type analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

soil_crop = pd.crosstab(df['soil_type'], df['recommended_crop'])
soil_crop.plot(kind='bar', stacked=True, ax=axes[0], colormap='viridis')
axes[0].set_title('Crops by Soil Type', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Soil Type')
axes[0].set_ylabel('Count')
axes[0].legend(bbox_to_anchor=(1.05, 1), fontsize=7)
axes[0].tick_params(axis='x', rotation=45)

# Box plot of pH by crop
df.boxplot(column='soil_ph', by='recommended_crop', ax=axes[1],
           rot=45, grid=False)
axes[1].set_title('Soil pH Distribution by Crop', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Crop')
axes[1].set_ylabel('pH')
plt.suptitle('')

plt.tight_layout()
plt.savefig('soil_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical summary per crop
crop_stats = df.groupby('recommended_crop')[numerical_cols].agg(['mean', 'std', 'min', 'max'])
print('\n📊 Statistical Summary per Crop (Mean values):')
print('='*80)
df.groupby('recommended_crop')[numerical_cols].mean().round(2)

---
## ⚙️ Step 3: Feature Engineering & Preprocessing

In [ ]:
# Encode categorical variables
le_soil = LabelEncoder()
le_region = LabelEncoder()
le_crop = LabelEncoder()

df['soil_encoded'] = le_soil.fit_transform(df['soil_type'])
df['region_encoded'] = le_region.fit_transform(df['region'])
df['crop_encoded'] = le_crop.fit_transform(df['recommended_crop'])

# Print label mappings
print('Soil Type Encoding:')
for i, label in enumerate(le_soil.classes_):
    print(f'  {i} → {label}')

print(f'\nCrop Encoding:')
for i, label in enumerate(le_crop.classes_):
    print(f'  {i} → {label}')

# Feature columns
feature_cols = ['soil_encoded', 'region_encoded', 'temperature_c', 'rainfall_mm',
                'humidity_pct', 'soil_ph', 'nitrogen_kg_ha', 'phosphorus_kg_ha',
                'potassium_kg_ha', 'farm_size_acres', 'altitude_m']

X = df[feature_cols].values
y = df['crop_encoded'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\n✅ Features prepared!')
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Number of features: {X_train.shape[1]}')
print(f'Number of crop classes: {len(le_crop.classes_)}')

---
## 🤖 Step 4: Model Training

### 4.1 Random Forest Classifier

In [ ]:
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

print('='*50)
print('RANDOM FOREST RESULTS')
print('='*50)
print(f'Accuracy:  {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)')
print(f'F1 Score:  {rf_f1:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, rf_pred, target_names=le_crop.classes_))

### 4.2 XGBoost Classifier

In [ ]:
# XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred, average='weighted')

print('='*50)
print('XGBOOST RESULTS')
print('='*50)
print(f'Accuracy:  {xgb_accuracy:.4f} ({xgb_accuracy*100:.2f}%)')
print(f'F1 Score:  {xgb_f1:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, xgb_pred, target_names=le_crop.classes_))

---
## 📈 Step 5: Model Evaluation & Comparison

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, pred, title in zip(axes, [rf_pred, xgb_pred],
                            ['Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGn', ax=ax,
                xticklabels=le_crop.classes_, yticklabels=le_crop.classes_)
    ax.set_title(f'{title} Confusion Matrix', fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.tick_params(axis='both', rotation=45)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-validation comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv = cross_val_score(rf_model, X_scaled, y, cv=cv, scoring='accuracy')
xgb_cv = cross_val_score(xgb_model, X_scaled, y, cv=cv, scoring='accuracy')

print('='*50)
print('5-FOLD CROSS-VALIDATION RESULTS')
print('='*50)
print(f'Random Forest:  {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')
print(f'XGBoost:        {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))
models = ['Random Forest', 'XGBoost']
accuracies = [rf_accuracy, xgb_accuracy]
f1_scores = [rf_f1, xgb_f1]
cv_means = [rf_cv.mean(), xgb_cv.mean()]

x = np.arange(len(models))
width = 0.25

bars1 = ax.bar(x - width, accuracies, width, label='Test Accuracy', color='#2d6a4f')
bars2 = ax.bar(x, f1_scores, width, label='F1 Score', color='#52b788')
bars3 = ax.bar(x + width, cv_means, width, label='CV Accuracy', color='#95d5b2')

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1.1)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, model, title in zip(axes, [rf_model, xgb_model],
                             ['Random Forest', 'XGBoost']):
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    sorted_features = [feature_cols[i] for i in indices]

    ax.barh(sorted_features[::-1], importances[indices][::-1], color='#40916c')
    ax.set_title(f'{title} Feature Importance', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔮 Step 6: Make Predictions

In [ ]:
def predict_crop(soil_type, region, temperature, rainfall, humidity,
                 soil_ph, nitrogen, phosphorus, potassium, farm_size, altitude,
                 model=xgb_model):
    """Predict best crop given farm conditions."""
    soil_enc = le_soil.transform([soil_type])[0]
    region_enc = le_region.transform([region])[0]

    features = np.array([[soil_enc, region_enc, temperature, rainfall, humidity,
                          soil_ph, nitrogen, phosphorus, potassium, farm_size, altitude]])
    features_scaled = scaler.transform(features)

    prediction = model.predict(features_scaled)
    probabilities = model.predict_proba(features_scaled)[0]

    predicted_crop = le_crop.inverse_transform(prediction)[0]

    # Top 3 recommendations
    top_3_idx = np.argsort(probabilities)[::-1][:3]
    top_3 = [(le_crop.inverse_transform([idx])[0], probabilities[idx]) for idx in top_3_idx]

    return predicted_crop, top_3

# Example prediction
print('='*50)
print('EXAMPLE PREDICTION')
print('='*50)
crop, top_3 = predict_crop(
    soil_type='Loam',
    region='Central Kenya (Nyeri, Murang\'a)',
    temperature=18,
    rainfall=1200,
    humidity=70,
    soil_ph=5.5,
    nitrogen=80,
    phosphorus=45,
    potassium=100,
    farm_size=5,
    altitude=1800
)

print(f'\n🌱 Best Crop: {crop}')
print(f'\nTop 3 Recommendations:')
for i, (c, prob) in enumerate(top_3, 1):
    print(f'  {i}. {c} — Confidence: {prob*100:.1f}%')

---
## 💾 Step 7: Export Models for Deployment

In [ ]:
# Save models and preprocessors
joblib.dump(rf_model, 'random_forest_model.pkl')
joblib.dump(xgb_model, 'xgboost_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le_soil, 'label_encoder_soil.pkl')
joblib.dump(le_region, 'label_encoder_region.pkl')
joblib.dump(le_crop, 'label_encoder_crop.pkl')

print('✅ All models and preprocessors saved!')
print('\nFiles exported:')
print('  • random_forest_model.pkl')
print('  • xgboost_model.pkl')
print('  • scaler.pkl')
print('  • label_encoder_soil.pkl')
print('  • label_encoder_region.pkl')
print('  • label_encoder_crop.pkl')
print('  • kenya_crop_dataset.csv')
print('\n📌 Upload these to your ICAS backend for production use.')

---
## 📋 Summary

| Metric | Random Forest | XGBoost |
|--------|--------------|----------|
| Test Accuracy | See above | See above |
| F1 Score | See above | See above |
| Cross-Val (5-fold) | See above | See above |

### Key Findings:
- **Rainfall** and **temperature** are the most important features for crop selection
- **Soil type** and **soil pH** significantly influence recommendations
- Both models achieve strong performance on the Kenyan agricultural dataset
- The pipeline is ready for integration with the ICAS web application

### Next Steps:
1. Collect real-world farmer data to replace synthetic dataset
2. Integrate handheld soil sensor readings
3. Add NDVI satellite vegetation data
4. Deploy models via ICAS backend API